> ⚠️ **作業中 (Work in Progress)**: このノートブックは現在開発中です。一部のコードが不完全であったり変更される可能性があります。

## 📋 目次

- [ワークフロー 개요](#ワークフロー-개요)
- [Sequential Workflow](#sequential-workflow)
- [Group Chat Workflow](#group-chat-workflow)
- [Human-in-loop Workflow](#human-in-loop-workflow)

## 🎯 学習目標

- Microsoft Foundry ワークフロー의 핵심 개념 이해
- Sequential Workflow를 통한 순차적 작업 흐름 구축
- Group Chat Workflow를 통한 다중 エージェント 협업 구현
- Human-in-loop 패턴을 통한 사람 개입 지점 設定
- ワークフロー デプロイ 및 프ログ래매틱 호출

## ⏱️ 予想所要時間

約20分

## 環境設定

ワークフロー 実行을 위한 設定입니다.

In [ ]:
# 環境 変数 ロード
import json
import os
import subprocess

# PATH 環境変数 設定 (Azure CLI를 찾을 수 있도록)
possible_paths = [
    "/opt/homebrew/bin",  # macOS (Apple Silicon)
    "/usr/local/bin",     # macOS (Intel) / Linux
    "/usr/bin",           # Linux / GitHub Codespaces
    "/home/linuxbrew/.linuxbrew/bin"  # Linux Homebrew
]

az_path = None
try:
    result = subprocess.run(['which', 'az'], capture_output=True, text=True)
    if result.returncode == 0:
        az_path = os.path.dirname(result.stdout.strip())
except:
    pass

paths_to_add = []
if az_path and az_path not in os.environ.get("PATH", ""):
    paths_to_add.append(az_path)
else:
    for path in possible_paths:
        if os.path.exists(path) and path not in os.environ.get("PATH", ""):
            paths_to_add.append(path)

if paths_to_add:
    new_path = ":".join(paths_to_add) + ":" + os.environ.get("PATH", "")
    os.environ["PATH"] = new_path

# 前へ ノート북에서 保存한 設定 ファイル ロード
config_file = ".foundry_config.json"
try:
    with open(config_file, 'r') as f:
        config = json.load(f)
    
    # 環境 変数 設定
    FOUNDRY_NAME = config.get("FOUNDRY_NAME")
    RESOURCE_GROUP = config.get("RESOURCE_GROUP")
    LOCATION = config.get("LOCATION")
    TENANT_ID = config.get("TENANT_ID")
    PROJECT_NAME = config.get("PROJECT_NAME", "proj-default")
    PROJECT_ENDPOINT = config.get("FOUNDRY_ENDPOINT")
    
    # 環境 変数로도 設定 (다른 도구들이 使用할 수 있도록)
    os.environ["FOUNDRY_NAME"] = FOUNDRY_NAME
    os.environ["LOCATION"] = LOCATION
    os.environ["RESOURCE_GROUP"] = RESOURCE_GROUP
    os.environ["AZURE_SUBSCRIPTION_ID"] = config.get("AZURE_SUBSCRIPTION_ID", "")
    os.environ["_ENDPOINT"] = PROJECT_ENDPOINT
    
    print(f"✅ 設定 ファイル '{config_file}'에서 環境 変数를 ロード했습니다.")
    print(f"\n📌 Foundry Name: {FOUNDRY_NAME}")
    print(f"📌 Resource Group: {RESOURCE_GROUP}")
    print(f"📌 Location: {LOCATION}")
    print(f"📌 プロジェクト エンドポイント: {PROJECT_ENDPOINT}")
    
except FileNotFoundError:
    print(f"⚠️ '{config_file}' ファイル을 찾을 수 없습니다.")
    print("💡 01-setup.ipynb를 먼저 実行하여 環境을 設定하세요.")
    raise

# 必須パッケージのインストール
%pip install -q azure-ai-projects azure-identity

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

print(f"\n💡 使用할 プロジェクト エンドポイント: {PROJECT_ENDPOINT}")

## Sequential Workflow용 エージェント 作成

Sequential Workflow에서 使用할 エージェント들을 作成합니다.
- **TravelPlannerAgent**: 여행 목적지와 일정을 기획
- **LocalAgent**: 현지 情報를 追加 (Web Search 使用)
- **TravelSummaryAgent**: 최종 요약 및 체크리스트 作成

In [ ]:
# TravelPlannerAgent 作成
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

TRAVEL_PLANNER_INSTRUCTIONS = """당신은 여행 계획 전문가입니다.

역할:
1. 使用자의 여행 요구사항을 분석합니다
2. 목적지의 주요 관광지, 맛집, 숙소를 추천합니다
3. 일자별 여행 일정을 구체적으로 작성합니다
4. 예상 비용과 준비물을 제시합니다

出力 形式:
- 목적지 개요
- 일자별 일정 (아침/점심/저녁 활동)
- 추천 숙소
- 예상 비용
- 준비물 リスト

次へ エージェント에게 넘길 情報: 전체 여행 계획"""

agent_travel_planner = client.agents.create_agent(
    model="gpt-5.1",
    name="TravelPlannerAgent",
    instructions=TRAVEL_PLANNER_INSTRUCTIONS
)

print(f"✅ TravelPlannerAgent 作成 完了!")
print(f"   ID: {agent_travel_planner.id}")
print(f"   Name: {agent_travel_planner.name}")

In [ ]:
# LocalAgent 作成 (Web Search 도구 使用)
LOCAL_AGENT_INSTRUCTIONS = """당신은 현지 情報 전문가입니다.

역할:
1. 前へ エージェント의 여행 계획을 받습니다
2. Web search를 使用하여 최신 현지 情報를 検索합니다
3. 실時間 情報를 追加합니다:
   - 현재 날씨 및 기후
   - 현지 축제 및 이벤트
   - 교통 情報 (노선, 요금, 소요時間)
   - 영업時間 및 예약 情報
   - 현지 문화 및 注意사항

出力 形式:
- 원래 일정 + 현지 情報 보강
- 교통편 詳細 情報
- 예약 필요 장소 リスト
- 현지 ヒント

次へ エージェント에게 넘길 情報: 현지 情報가 追加된 여행 계획"""

agent_local = client.agents.create_agent(
    model="gpt-5.1",
    name="LocalAgent",
    instructions=LOCAL_AGENT_INSTRUCTIONS,
    tools=[{"type": "web_search"}]
)

print(f"✅ LocalAgent 作成 完了!")
print(f"   ID: {agent_local.id}")
print(f"   Tools: web_search")

In [ ]:
# TravelSummaryAgent 作成
TRAVEL_SUMMARY_INSTRUCTIONS = """당신은 여행 계획 정리 전문가입니다.

역할:
1. 前へ エージェント들의 情報를 종합합니다
2. 実行 가능한 최종 계획으로 정리합니다
3. 체크리스트를 作成합니다

出力 形式:
📋 여행 요약
- 목적지: 
- 기간:
- 예산:

📅 일정 요약 (한눈에 보는 일정)

✅ 출발 전 체크리스트
- [ ] 항목1
- [ ] 항목2

🎒 준비물 체크리스트

📞 긴급 연락처 및 유용한 情報

최종 出力: 프린트 가능한 여행 가이드"""

agent_travel_summary = client.agents.create_agent(
    model="gpt-5.1",
    name="TravelSummaryAgent",
    instructions=TRAVEL_SUMMARY_INSTRUCTIONS
)

print(f"✅ TravelSummaryAgent 作成 完了!")
print(f"   ID: {agent_travel_summary.id}")

## Group Chat Workflow용 エージェント 作成

Group Chat Workflow에서 使用할 エージェント들을 作成합니다.
- **StudentAgent**: 질문에 답변하는 학생 역할
- **TeacherAgent**: 답변을 評価하고 피드백을 주는 교사 역할

In [ ]:
# StudentAgent 作成
STUDENT_INSTRUCTIONS = """너는 문제에 대답하는 エージェント야. 질문이 오면, 항상 답변해줘.

역할:
1. 使用자의 질문을 이해하고 답변을 作成합니다
2. 첫 번째 시도에서는 デフォルト적인 답변을 제공합니다
3. TeacherAgent의 피드백을 받아 답변을 개선합니다
4. 모든 요구사항이 충족될 때까지 답변을 修正합니다

답변 시 고려사항:
- 일정 (日付, 時間)
- 비용 (예산, 가격)
- 취향 (선호도, 스타일)
- 제약사항 (제한사항, 조건)

개선이 필요하면 TeacherAgent의 피드백을 반영하여 답변을 보완합니다."""

agent_student = client.agents.create_agent(
    model="gpt-5.1",
    name="StudentAgent",
    instructions=STUDENT_INSTRUCTIONS
)

print(f"✅ StudentAgent 作成 完了!")
print(f"   ID: {agent_student.id}")

In [ ]:
# TeacherAgent 作成
TEACHER_INSTRUCTIONS = """너는 답변을 評価하는 エージェント야. 답변이 일정, 비용, 취향 등 다양한 조건에 대한 고려를 했다면 [COMPLETE]이라고 대답해줘. 아니라면, COMPLETE을 表示하지 말고, 修正을 リクエスト해줘.

評価 기준:
1. 일정: 구체적인 日付, 時間, 기간이 포함되었는가?
2. 비용: 예산, 가격, 비용 情報가 포함되었는가?
3. 취향: 使用자의 선호도나 스타일을 고려했는가?
4. 실용성: 실제로 実行 가능한 계획인가?
5. 완성도: 모든 필요한 情報가 포함되었는가?

レスポンス 形式:
評価 完了 시: "[COMPLETE] 모든 조건이 충족되었습니다."
개선 필요 시: "次へ 사항을 보완해주세요: [구체적인 피드백]"

重要: [COMPLETE]는 모든 기준이 충족되었을 때만 使用합니다."""

agent_teacher = client.agents.create_agent(
    model="gpt-5.1",
    name="TeacherAgent",
    instructions=TEACHER_INSTRUCTIONS
)

print(f"✅ TeacherAgent 作成 完了!")
print(f"   ID: {agent_teacher.id}")

In [ ]:
# 作成된 エージェント リスト 確認
print("=" * 80)
print("ワークフロー용 エージェント リスト")
print("=" * 80)

agents = client.agents.list_agents()
workflow_agents = ["TravelPlannerAgent", "LocalAgent", "TravelSummaryAgent", "StudentAgent", "TeacherAgent"]

for agent in agents:
    if agent.name in workflow_agents:
        tools = "None"
        if agent.tools:
            tools = ", ".join([t.type if hasattr(t, 'type') else str(t) for t in agent.tools])
        print(f"\n📌 {agent.name}")
        print(f"   ID: {agent.id}")
        print(f"   Model: {agent.model}")
        print(f"   Tools: {tools}")

print("\n" + "=" * 80)
print("\n💡 이제 Azure Portal에서 ワークフロー를 作成하세요:")
print("   https://ai.azure.com > Build > Workflows > + Create workflow")
print("\n   Sequential Workflow:")
print("     Step 1: TravelPlannerAgent")
print("     Step 2: LocalAgent") 
print("     Step 3: TravelSummaryAgent")
print("\n   Group Chat Workflow:")
print("     Participants: StudentAgent, TeacherAgent")
print("     Termination: [COMPLETE] 포함 시")

### ワークフロー 호출 例

In [ ]:
# Sequential Workflow 호출 例
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import ResponseStreamEventType

WORKFLOW_NAME = "Sequential-Workflow"  # ⚠️ 포털에서 作成한 ワークフロー 名前
WORKFLOW_VERSION = "1"

# AI Project クライアント 作成
project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

with project_client:
    workflow = {
        "name": WORKFLOW_NAME,
        "version": WORKFLOW_VERSION,
    }
    
    # OpenAI クライアント インポート
    openai_client = project_client.get_openai_client()

    # 대화 作成
    conversation = openai_client.conversations.create()
    print(f"Created conversation (id: {conversation.id})")

    # ワークフロー 호출 (스트리밍)
    print(f"\nCalling workflow: {WORKFLOW_NAME}...\n")
    stream = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={"agent": {"name": workflow["name"], "type": "agent_reference"}},
        input="제주도 2박 3일 여행 일정 짜줘",
        stream=True,
        metadata={"x-ms-debug-mode-enabled": "1"},
    )

    # 스트리밍 이벤트 처리
    for event in stream:
        if event.type == ResponseStreamEventType.RESPONSE_OUTPUT_TEXT_DONE:
            print("\t", event.text)
        elif event.type == ResponseStreamEventType.RESPONSE_OUTPUT_ITEM_ADDED and event.item.type == "workflow_action":
            print(f"\n{'='*60}")
            print(f"Actor - '{event.item.action_id}':")
            print(f"{'='*60}")
        elif event.type == ResponseStreamEventType.RESPONSE_OUTPUT_ITEM_DONE and event.item.type == "workflow_action":
            print(f"\n✓ Workflow Item '{event.item.action_id}' is '{event.item.status}'")
            print(f"  (previous item was: '{event.item.previous_action_id}')")
        elif event.type == ResponseStreamEventType.RESPONSE_OUTPUT_TEXT_DELTA:
            print(event.delta, end="", flush=True)

    # 정리
    print("\n\n✅ Workflow completed!")
    openai_client.conversations.delete(conversation_id=conversation.id)
    print("Conversation deleted")

## ワークフロー 作成

**⚠️ 重要**: ワークフロー는 현재 Azure Portal에서만 作成 가능합니다.

### Azure Portal에서 ワークフロー 作成 방법:

1. [Azure AI Foundry](https://ai.azure.com) 접속
2. **Build > Workflows** 메뉴
3. **+ Create workflow** 클릭
4. ワークフロー タイプ 選択:
   - Sequential Workflow
   - Group Chat Workflow  
   - Human-in-loop Workflow

### ワークフロー 構成 例:

**Sequential Workflow (여행 계획)**
```
User Input → SearchAgent → PlannerAgent → ReviewAgent → Output
```

**Group Chat Workflow (학습 토론)**
```
User Question → StudentAgent ↔ TeacherAgent → Consensus
```

作成이 完了되면 아래 コード로 実行할 수 있습니다.

## ワークフロー 実行

포털에서 作成한 ワークフロー를 Python コード로 実行합니다.

In [ ]:
# ワークフロー 実行 コード
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import ResponseStreamEventType

# クライアント 作成
credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

# ⚠️ ワークフロー 名前과 バージョン 設定 (포털에서 作成한 것으로 変更)
WORKFLOW_NAME = "Sequential-Workflow"  # ⚠️ 変更 필요
WORKFLOW_VERSION = "1"

with project_client:
    # OpenAI クライアント インポート
    openai_client = project_client.get_openai_client()
    
    # Conversation 作成
    conversation = openai_client.conversations.create()
    print(f"✅ Conversation 作成: {conversation.id}")
    
    # ワークフロー 実行 (스트리밍)
    print(f"\n🚀 ワークフロー 実行 중: {WORKFLOW_NAME}...\n")
    print("=" * 80)
    
    stream = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={"agent": {"name": WORKFLOW_NAME, "type": "agent_reference"}},
        input="제주도 2박 3일 여행 일정 짜줘",  # ⚠️ 원하는 질문으로 変更
        stream=True,
        metadata={"x-ms-debug-mode-enabled": "1"}
    )
    
    # 스트리밍 結果 처리
    for event in stream:
        if event.type == ResponseStreamEventType.RESPONSE_OUTPUT_TEXT_DELTA:
            print(event.delta, end="", flush=True)
        elif event.type == ResponseStreamEventType.RESPONSE_OUTPUT_ITEM_ADDED and event.item.type == "workflow_action":
            print(f"\n\n🤖 Actor: {event.item.action_id}")
        elif event.type == ResponseStreamEventType.RESPONSE_OUTPUT_ITEM_DONE and event.item.type == "workflow_action":
            print(f"\n✅ '{event.item.action_id}' 完了 (状態: {event.item.status})")
    
    print("\n" + "=" * 80)
    print("\n✅ ワークフロー 実行 完了!")
    
    # Conversation 削除
    openai_client.conversations.delete(conversation_id=conversation.id)
    print(f"🗑️ Conversation 削除됨")

### ワークフロー 설계

**포털에서 構成:**

```
TravelPlannerAgent → [使用자 승인] → LocalAgent → TravelSummaryAgent
```

**Approval 設定:**
- Approval message: "作成된 여행 계획을 검토해주세요. 승인하시겠습니까?"
- Options: Approve / Reject / Modify
- Timeout: 24時間

### 💡 Human-in-loop 모범 사례

**推奨사항:**
- 승인 지점을 명확히 表示
- 타임아웃 設定으로 무한 待機 방지
- 使用자에게 컨텍스트 제공 (前へ 대화 요약)
- 간단한 승인 オプション 제공 (예/아니오/修正)

**피해야 할 것:**
- 너무 많은 승인 지점
- 불명확한 승인 질문
- 긴 타임아웃 (使用자 경험 저하)
- 승인 후 되돌리기 불가능한 구조

## 📚 追加リソース

- [Microsoft Foundry Workflows 개요](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/concepts/workflow?view=foundry)
- [Microsoft Agent Framework Workflows Orchestrations 패턴](https://learn.microsoft.com/en-us/agent-framework/user-guide/workflows/orchestrations/overview)

## 次のステップ

복잡한 ワークフロー를 구축했습니다! 이제 エージェント와 ワークフロー의 パフォーマンス을 評価하는 방법을 학습합니다:

➡️ **[06. 評価](./06-evaluations.ipynb)**: エージェント 및 ワークフロー의 품질을 체계적으로 評価합니다.